# Wikipedia Retriever

In [1]:
from langchain_community.retrievers import WikipediaRetriever

In [3]:
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [4]:
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

In [5]:
docs = retriever.invoke(query)

In [6]:
docs

[Document(metadata={'title': 'India–Iran relations', 'summary': "India–Iran relations are the bilateral relationship between the Republic of India and the Islamic Republic of Iran. Independent India and Iran established diplomatic relations on 15 March 1950.\nContact between both ancient Persia and ancient India date to ancient times, and can be seen through the diffusion of Persian culture among Islamic culture in much of South Asia; furthermore, around 15% of the Muslims in India are Shia, a group Iran considers itself to represent on the world stage. Outside the Islamic community, the impact of Persian culture has primarily been in Northwest India.\nDuring much of the Cold War, relations between India and the erstwhile Imperial State of Iran suffered due to their differing political interests: India endorsed a non-aligned position but fostered strong links with the Soviet Union, while Iran was an open member of the Western Bloc and enjoyed close ties with the United States. While In

In [7]:
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1}---")
    print(f"Content:\n{doc.page_content}...")


--- Result 1---
Content:
India–Iran relations are the bilateral relationship between the Republic of India and the Islamic Republic of Iran. Independent India and Iran established diplomatic relations on 15 March 1950.
Contact between both ancient Persia and ancient India date to ancient times, and can be seen through the diffusion of Persian culture among Islamic culture in much of South Asia; furthermore, around 15% of the Muslims in India are Shia, a group Iran considers itself to represent on the world stage. Outside the Islamic community, the impact of Persian culture has primarily been in Northwest India.
During much of the Cold War, relations between India and the erstwhile Imperial State of Iran suffered due to their differing political interests: India endorsed a non-aligned position but fostered strong links with the Soviet Union, while Iran was an open member of the Western Bloc and enjoyed close ties with the United States. While India did not welcome the 1979 Islamic Revo

 # Vector Store Retriever

In [33]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

In [9]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [11]:
embeddings = OllamaEmbeddings(model="llama3")

#Create Chroma vector store in memory
vectorize = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="collection"
)

In [12]:
# Convert vectorestore into a retriever

retriever = vectorize.as_retriever(search_kwargs={"k":2})

In [13]:
query = "what is chroma used for?"
results = retriever.invoke(query)

In [14]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search.

--- Result 2 ---
Chroma is a vector database optimized for LLM-based search.


In [15]:
results = vectorize.similarity_search(query, k=2)

# MMR

In [17]:
# sample documents

docs = [
     Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [21]:
from langchain_community.vectorstores import FAISS

# Initialize Ollama Embeddings
embedding_model = OllamaEmbeddings(model="llama3")

# Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [23]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5}  #k = top results, lambda_mult = relevance-diversity balance
)

In [24]:
query = "What is LangChain?"
results = retriever.invoke(query)

In [25]:
for i, doc in enumerate(results):
    print(f"--- Results {i+1} ---")
    print(doc.page_content)

--- Results 1 ---
LangChain is used to build LLM based applications.
--- Results 2 ---
Embeddings are vector representations of text.
--- Results 3 ---
MMR helps you get diverse results when doing similarity search.


# Multi Query Retriever

In [26]:
from langchain_classic.retrievers import MultiQueryRetriever

In [27]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [28]:
embeddings = OllamaEmbeddings(model="llama3")

In [29]:
vectorStore = FAISS.from_documents(documents=all_docs, embedding=embeddings)

In [30]:
similarity_retriever = vectorStore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [34]:
mutltiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorStore.as_retriever(search_kwargs={"k": 5}),
    llm=ChatOllama(model="llama3")
)

In [35]:
query = "How to improve energy levels and maintain balance?"

In [36]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= mutltiquery_retriever.invoke(query)

In [37]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 2 ---
Deep sleep is crucial for cellular repair and emotional regulation.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 5 ---
Regular walking boosts heart health and can reduce symptoms of depression.
******************************************************************************************************************************************************

--- Result 1 ---
Python balances readability with power, making it a popular system design language.

--- Result 2 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 4 ---
Black holes bend spacetime and store immense gravitational energy.


# Contextual Compression Retriever

In [38]:
from langchain_classic.retrievers import ContextualCompressionRetriever

In [39]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [40]:
embeddings = OllamaEmbeddings(model="llama3")

vectorStore = FAISS.from_documents(
    documents=docs,
    embedding=embeddings
)

In [41]:
base_retriever = vectorStore.as_retriever(search_kwargs={"k": 5})

In [42]:
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3")

In [43]:
compressor = LLMChainExtractor.from_llm(llm)

In [44]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [45]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [46]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Extracted relevant parts:

Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The extracted relevant part is:

"The chlorophyll in plant cells captures sunlight during photosynthesis."

This is the only part of the context that directly answers the question about photosynthesis.
